# 第 1 课｜从招聘数据库到可分析的数据集

**90 分钟：问题定义 15′ → Python/Pandas 25′ → SQLite/JSON 35′ → 质量检查 15′**

本课结束，你能解释“一行是什么”、读取 SQLite、拆开 JSON，并交付可追溯的职位明细表。

业务任务：描述这批官网职位样本的公司、地点与招聘类型结构，为技能分析做准备。
数据分析是“提出问题 → 定义指标 → 整理数据 → 比较与解释 → 支持决策”，不等于画图。

本项目采集工作已完成；你的任务是分析。职位条目不是招聘人数；历史保留、来源覆盖差异、
队列同步删除都会影响样本。我们分析的是固定快照中的待同步记录，不能称为当前全网职位总量。

**运行顺序**：四本 Notebook 按 01→04 执行，每本均从顶部运行。教师版已保留真实执行输出。

> **增强版阅读方法**：先看本节问题，预测输出，再运行代码，最后解释数字。
> 关键代码已加中文注释，说明“为什么做、参数是什么、结果的行列单位是什么”。
> 标有【主线】的段落用于课堂；【拓展】用于教师演示或课后，不要求90分钟内完成所有扩展。
> 长代码按完整步骤拆成较短单元；每本仍需从上到下依次运行。

In [1]:
# Path把文件路径当作对象处理；用 / 拼接路径可兼容不同操作系统。
from pathlib import Path
import sys, json, re
# NumPy处理数组和数值；np只是约定俗成的简称，不是Python关键字。
import numpy as np
# Pandas处理带行列标签的表格；Series是一列，DataFrame是一张表。
import pandas as pd
# display显示表格，Markdown把动态生成的结论渲染成Notebook文字。
from IPython.display import display, Markdown

# 支持从项目根目录、课程目录或 notebooks 目录启动。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'course_utils.py').exists()), None)
if ROOT is None:
    ROOT = next((p / 'data_analysis202612' for p in candidates
                 if (p / 'data_analysis202612' / 'course_utils.py').exists()), None)
assert ROOT is not None, '请从项目根目录或课程目录启动 Jupyter'
# 将课程目录加入模块搜索路径，才能导入同目录的course_utils.py。
sys.path.insert(0, str(ROOT))
# 教学工具只封装路径、清洗和绘图；SKILLS是词典，KEY是复合身份字段，SEED固定随机种子。
from course_utils import *
# 以下设置只影响屏幕展示，不删数据、不改变统计结果；完整结果仍会导出。
pd.set_option('display.max_columns', 14)
pd.set_option('display.max_colwidth', 65)
pd.set_option('display.max_rows', 20)
print('课程目录：', ROOT)
print('Python：', sys.version.split()[0])
print('解释器：', sys.executable)

课程目录： /Users/luosen/WorkBuddy/JobsCrawler/data_analysis202612
Python： 3.10.16
解释器： /Users/luosen/WorkBuddy/JobsCrawler/data_analysis202612/.venv/bin/python


## 1. 必要的 Python：字典、列表、函数、DataFrame

字典像一条职位，列表像多条职位；函数把重复规则封装起来。
DataFrame 是有行列标签的表格，`loc` 筛选、`groupby` 分组、`agg` 聚合。
下面用小数据先练习，再处理真实数据。缺失值不是零。

In [2]:
# 列表的每个元素是一条职位字典；字典的键对应字段名，值对应该字段内容。
examples = [
    {'title': '数据分析师', 'city': '北京', 'years': 2},
    {'title': '算法实习生', 'city': '上海', 'years': None},
    {'title': '商业分析师', 'city': '北京', 'years': 3},
]
# 定义可重复调用的函数；传入一条字典和关键词，返回True或False。
def title_contains(job, keyword):
    # 方括号按字典键取值；in检查子字符串，不等于语义相似度判断。
    return keyword in job['title']

print([x['title'] for x in examples if title_contains(x, '分析')])
# 将“字典列表”转为表格：每个字典成为一行，同名键对齐成一列。
demo = pd.DataFrame(examples)
display(demo)
display(demo.groupby('city', as_index=False).agg(岗位数=('title', 'size'), 已知年限均值=('years', 'mean')))
# loc[行条件, 列名列表]同时筛行与选列；ge(2)表示>=2，缺失值不会被当作满足条件。
display(demo.loc[demo['years'].ge(2), ['title', 'city']])

['数据分析师', '商业分析师']


,title,city,years
0,数据分析师,北京,2.0
1,算法实习生,上海,NaN
2,商业分析师,北京,3.0


,city,岗位数,已知年限均值
0,上海,1,NaN
1,北京,2,2.5


,title,city
0,数据分析师,北京
2,商业分析师,北京


**练习 1（3 分钟）**：统计示例中经验年限缺失的数量；把筛选条件改成“岗位名称包含分析”。
预测 `size` 与 `count` 在缺失值存在时会不会相同，再运行验证。

In [3]:
# 参考解答：课堂可先折叠本单元，学生在上方新增代码单元练习。
print('年限缺失数：', demo['years'].isna().sum())
display(demo.loc[demo['title'].str.contains('分析', na=False)])
display(demo.groupby('city')['years'].agg(['size', 'count']))
assert demo['years'].isna().sum() == 1

年限缺失数： 1


,title,city,years
0,数据分析师,北京,2.0
2,商业分析师,北京,3.0


,size,count
city,,
上海,1,0
北京,2,2


## 1A. 【主线】从业务问题到分析字段：先写一条可计算的问题

“招聘市场怎么样”无法直接计算。把它改写为：
**在这份固定采集样本中，各公司社招职位的城市覆盖有何不同？**

| 部分 | 本课约定 | 常见错误 |
|---|---|---|
| 对象 | 固定快照中的职位条目 | 把职位条目当作招聘人数 |
| 范围 | 明确公司、招聘类型、采集快照 | 忽略历史保留或渠道缺失 |
| 维度 | source、recruitment_type、cities | 用同名但不同口径的字段混比 |
| 指标 | 职位数、覆盖率、字段缺失率 | 先画图，再临时寻找解释 |
| 输出 | 底稿、图表、证据和限制 | 只展示一张没有分母的百分比图 |

先练Python必要知识：`str/int/float/bool/list/dict/None`；变量是对象的名字，
`=`是赋值、`==`是比较。`.get()`在键缺失时返回备用值；`[]`取不存在的键会抛错。

In [4]:
# 这是一条教学示例，不是新增的真实职位；用它观察类型和安全取值。
sample_job = {'title': '数据分析师', 'cities': ['北京', '上海'], 'salary': None}
for field, value in sample_job.items():
    print(field, '→', repr(value), '→', type(value).__name__)

# get只处理“键不存在”；salary键存在且为None时，不会自动得到备用数字。
print('安全读取未知字段：', sample_job.get('unknown', '未提供'))
assert sample_job.get('salary', 0) is None

# 用清晰的中间变量构建布尔掩码，便于解释每一步，而非写一长串嵌套表达式。
is_analysis = demo['title'].str.contains('分析', na=False)
has_known_years = demo['years'].notna()
display(demo.loc[is_analysis & has_known_years, ['title', 'years']])

# Series条件用&/|/~逐元素组合，每个比较加括号；不要使用Python标量and/or。
mask = (demo['years'] >= 2) & (demo['city'] == '北京')
assert mask.sum() == 2
display(demo.loc[mask])

title → '数据分析师' → str
cities → ['北京', '上海'] → list
salary → None → NoneType
安全读取未知字段： 未提供


,title,years
0,数据分析师,2.0
2,商业分析师,3.0


,title,city,years
0,数据分析师,北京,2.0
2,商业分析师,北京,3.0


## 1B. 【主线】行索引、位置和类型转换

`loc`根据行标签选择，`iloc`根据第几行选择。筛选后旧索引不会自动重排，
这正是合并表格时发生错位的常见原因。`pd.to_numeric(errors='coerce')`把不合法数值留作缺失，
不能把“面议”转成0后计算平均薪资。

**先预测再运行**：索引改成101/205/309后，第一行的标签和位置分别是什么？

In [5]:
# 复制演示表，不改动之前单元的demo；loc与iloc可以指向相同的行，但含义不同。
indexed_demo = demo.copy()
indexed_demo.index = [101, 205, 309]
display(indexed_demo.loc[[101], ['title', 'city']])
display(indexed_demo.iloc[[0], [0, 1]])
assert indexed_demo.loc[101, 'title'] == indexed_demo.iloc[0]['title']

# 示例工资仅用于展示脏值转换；它们不是本项目的真实薪资统计。
raw_numbers = pd.Series(['10000', '面议', None, '20000'])
numeric_values = pd.to_numeric(raw_numbers, errors='coerce')
display(pd.DataFrame({'原始值': raw_numbers, '转换后': numeric_values}))
print('忽略缺失的均值：', numeric_values.mean())
print('错误补0后的均值：', numeric_values.fillna(0).mean())
assert numeric_values.isna().sum() == 2

,title,city
101,数据分析师,北京


,title,city
101,数据分析师,北京


,原始值,转换后
0,10000,10000.0
1,面议,NaN
2,None,NaN
3,20000,20000.0


忽略缺失的均值： 15000.0
错误补0后的均值： 7500.0


## 2. 建立一致、只读的原始快照

直接复制 `.db` 可能遗漏 WAL 中已提交的记录。这里用 SQLite backup API 读取原库，
生成课程快照，并记录 SHA-256、创建时间、表行数和完整性校验。已有快照不会自动覆盖。
原库不执行写入，后续课程只读取这份固定快照。复制过程由 `course_utils.py` 封装。

In [6]:
# 生成/复用一致性快照，并返回路径与台账两个对象；SHA校验使重跑仍使用同一份输入。
snapshot, manifest = snapshot_database()
display(pd.Series(manifest, name='数据接收台账'))
import sqlite3
# mode=ro强制只读；with管理连接事务作用域，查询不会修改原始采集数据。
with sqlite3.connect(snapshot.as_uri() + '?mode=ro', uri=True) as conn:
    # sqlite_master保存数据库对象定义；先看真实表结构，再决定读取哪些列。
    schema = pd.read_sql_query("SELECT name, sql FROM sqlite_master WHERE type='table'", conn)
    # SQL结果直接读成DataFrame；按seq排序，使后续抽样和文本行号对应可重现。
    raw = pd.read_sql_query('SELECT * FROM pending_jobs ORDER BY seq', conn)
display(schema)
print('快照职位数：', len(raw), '；原始列数：', len(raw.columns))
assert len(raw) == manifest['table_rows']['pending_jobs']

source                   /Users/luosen/WorkBuddy/JobsCrawler/state/pending_queue.db
snapshot           /Users/luosen/WorkBuddy/JobsCrawler/data_analysis202612/data/...
created_at                                         2026-10-28T17:07:40.124351+08:00
method                    SQLite backup API; source mode=ro; includes committed WAL
sha256             29efa641bd5db92362ae97b75e226b994e5148a2bfe8cb442e397c0860f413ac
size_bytes                                                                335814656
schema_version                                                                    5
integrity_check                                                                  ok
table_rows         {'pending_jobs': 40661, 'sqlite_sequence': 1, 'job_fingerprin...
Name: 数据接收台账, dtype: object

,name,sql
0,pending_jobs,CREATE TABLE pending_jobs (\n seq INTEGER PRIM...
1,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"
2,job_fingerprints,"CREATE TABLE job_fingerprints (\n source TEXT,..."
3,job_organizations,CREATE TABLE job_organizations (\n source TEXT...
4,api_cache,"CREATE TABLE api_cache(cache_key TEXT PRIMARY KEY,version TEX..."
5,recruitment_batches,"CREATE TABLE recruitment_batches(source TEXT,batch_id TEXT,re..."


快照职位数： 40661 ；原始列数： 9


## 2A. 【主线】SQL与Pandas对照：同一问题，两种表达

SQL先在数据库执行筛选与聚合，适合少读数据；Pandas适合进入内存后的灵活转换。
`WHERE`对应布尔筛选、`GROUP BY`对应`groupby`、`ORDER BY`对应`sort_values`。
SQL参数使用占位符`?`传入，不把用户输入拼接成SQL。

本节只比较**队列source层面的条数**，与payload字段清洗无关。

In [7]:
# 数据库侧计算公司条数：无需先把每条payload展开。
# mode=ro强制只读；with管理连接事务作用域，查询不会修改原始采集数据。
with sqlite3.connect(snapshot.as_uri() + '?mode=ro', uri=True) as conn:
    sql_counts = pd.read_sql_query(
        'SELECT source, COUNT(*) AS n FROM pending_jobs GROUP BY source ORDER BY source', conn)
    sql_sample = pd.read_sql_query(
        'SELECT seq, source, portal, job_id FROM pending_jobs WHERE source = ? ORDER BY seq LIMIT ?',
        conn, params=('tencent', 3))

# 内存侧回答同一问题。两边先统一行顺序与索引，避免“值相同、索引不同”的假失败。
pandas_counts = raw.groupby('source').size().rename('n').reset_index().sort_values('source').reset_index(drop=True)
pd.testing.assert_frame_equal(sql_counts, pandas_counts)
display(sql_counts)
display(sql_sample)
print('SQL与Pandas公司计数完全一致')

,source,n
0,alibaba,4744
1,baidu,1058
2,bilibili,872
3,bytedance,17053
4,ctrip,534
5,didi,1808
6,iflytek,841
7,jd,2025
8,kuaishou,2563
9,meituan,3221


,seq,source,portal,job_id
0,17054,tencent,society,2055186895503273984
1,17055,tencent,society,2109891198515855360
2,17056,tencent,society,2070158764278984704


SQL与Pandas公司计数完全一致


## 2B. 【主线】观察一条JSON：字符串 → 字典 → 顶层表格

JSON中的`null/true/false`对应Python的`None/True/False`。
对象通常展开为列，数组通常保留为列表，直到决定统计单位时才展开为行。
下一单元的批量处理正是把这里的一条扩展到全部职位。

In [8]:
# 一次只查看第一条的结构；不把数百KB原文全部打印到屏幕。
one_text = raw.iloc[0]['payload']
one_object = json.loads(one_text)
print('解析前：', type(one_text).__name__, '；解析后：', type(one_object).__name__)
one_fields = pd.DataFrame([
    {'字段': key, 'Python类型': type(value).__name__, '值预览': repr(value)[:100]}
    for key, value in one_object.items()
])
display(one_fields.head(15))

# 对比两种展开深度：嵌套字段越深，列越多；列数增加并不意味着分析价值一定增加。
top_level = pd.json_normalize([one_object], max_level=0)
deep_level = pd.json_normalize([one_object])
display(pd.DataFrame({'方案': ['只展开顶层', '递归展开对象'],
                      '行数': [len(top_level), len(deep_level)],
                      '列数': [top_level.shape[1], deep_level.shape[1]]}))
assert len(top_level) == len(deep_level) == 1

解析前： str ；解析后： dict


,字段,Python类型,值预览
0,address,str,'中国大陆北京市海淀区北三环西路甲18号院大钟寺广场1号楼，邮编：100098'
1,address_list,list,"[{'active_status': 1, 'city': {'city_code': 'CT_11', 'en_name..."
2,batch_id,str,''
3,batch_name,NoneType,None
4,business_unit,NoneType,None
5,business_units,list,[]
6,city_code,str,'CT_11'
7,city_en_name,str,'Beijing'
8,city_list,list,"[{'code': 'CT_11', 'en_name': 'Beijing', 'i18n_name': '北京', '..."
9,city_name,str,'北京'


,方案,行数,列数
0,只展开顶层,1,48
1,递归展开对象,1,164


## 3. 展开 payload，并防止同名字段相互覆盖

`json.loads` 把 JSON 文本变为字典；`json_normalize(max_level=0)` 展开顶层字段，
保留数组/对象。若直接与队列表横向合并，`source`、`job_id` 会重名，所以先加 `payload.` 前缀。
坏 JSON 单独隔离，身份键不一致则停止处理。`portal` 是渠道标识，不强行等同于 `recruitment_type`。

In [9]:
# 分别存有效JSON、原始行索引和错误日志；坏行隔离后仍能按seq回查。
records, valid_indices, rejected = [], [], []
# items()同时给出行索引和值；解析不是对字符串按字符循环。
for idx, value in raw['payload'].items():
    try:
        # loads把JSON字符串还原为Python对象；它与读取文件的json.load不同。
        parsed = json.loads(value)
        # 能解析的JSON也可能是列表或数字；只有字典才符合“一条职位”的结构约定。
        if not isinstance(parsed, dict):
            raise ValueError('payload 不是对象')
        records.append(parsed)
        valid_indices.append(idx)
    # 只处理预期的解析/类型问题，并记录原因；不使用空except吞掉所有程序错误。
    except (ValueError, TypeError) as exc:
        rejected.append({'seq': int(raw.loc[idx, 'seq']), 'reason': str(exc)})

# max_level=0只展开顶层；否则raw_data等嵌套对象可能产生大量难以解释的列。
payload = pd.json_normalize(records, max_level=0)
# 让解析表恢复原始行索引；若中间隔离了一行，仍可正确对齐，不发生错位拼接。
payload.index = valid_indices
# join按索引对齐；payload.前缀防止source、job_id等同名列覆盖队列身份。
flat = raw.loc[valid_indices].drop(columns='payload').join(payload.add_prefix('payload.'))
# 队列与正文都带身份字段，先做一致性检查，再决定分析表采用哪一份。
identity_mismatches = {}
# 不把portal与recruitment_type直接比较：两者可能分别是渠道名与统一招聘类型。
for col in ['source', 'job_id', 'batch_id', 'source_site']:
    # 身份比较先统一空值与字符串类型；批次缺失的约定是空字符串，不是数字0。
    a = flat[col].fillna('').astype(str)
    # 给正文身份使用与队列相同的空值约定和类型，然后按行比较，避免None与空串产生假冲突。
    b = flat['payload.' + col].fillna('').astype(str)
    # ne表示不相等；布尔值求和统计冲突条数，int转成可写JSON的Python整数。
    identity_mismatches[col] = int(a.ne(b).sum())

In [10]:
# assert是数据契约：发现不一致就停止，而不是覆盖冲突后继续生成“正常”报表。
assert not any(identity_mismatches.values()), identity_mismatches
# 同一job_id可能在不同公司出现；必须以五字段复合身份检查重复。
assert not flat.duplicated(KEY).any(), '完整职位身份键重复'

# 保留队列身份列；raw_data 原文仍在快照，不复制到轻量分析表。
business_cols = [c for c in payload.columns if c not in ['source', 'job_id', 'batch_id', 'source_site', 'raw_data']]
# 显式copy创建独立分析表，后续增加列不会意外改动原始表的视图。
jobs = flat[list(raw.columns.drop('payload'))].copy()
# 逐个补充业务字段；队列身份保留，raw_data仍可从原始快照回溯。
for col in business_cols:
    jobs[col] = flat['payload.' + col]
# 重建连续行索引；drop=True避免把旧索引误当业务字段写入。
jobs = jobs.reset_index(drop=True)
jobs['job_key'] = jobs['seq'].astype(str)  # 仅在本固定快照内使用的便捷行键
display(jobs[['job_key', 'source', 'portal', 'title', 'city_name', 'recruitment_type']].head())
print('payload 顶层字段数：', len(payload.columns), '；隔离记录数：', len(rejected))

,job_key,source,portal,title,city_name,recruitment_type
0,1,bytedance,society,用户产品经理（造世界方向） - 抖音社交与互动,北京,society
1,2,bytedance,society,游戏资深3D场景模型设计师 - 心光工作室,杭州,society
2,3,bytedance,society,游戏音乐设计师（PVE射击项目） - 研发支持中心,上海,society
3,4,bytedance,society,芯片综合/STA工程师（存算与AI） - 芯片研发,北京,society
4,5,bytedance,society,东南亚跨境IP及主题活动运营-TikTok Shop,上海,society


payload 顶层字段数： 48 ；隔离记录数： 0


## 4. 保留原文，新增清洗列

清洗不是把空值随意补齐。这里去掉 HTML、规范空白，对标题缺失做标记；
学历、薪资保留缺失，不推测。原始城市字段保留，清洗后的地点标签另存。

真实差异：字节城市列表含字典，百度可能一个元素含多个城市，讯飞使用“安徽省·合肥市”，
京东有省级地点，携程有英文城市。城市映射是可审计规则，不是完美地理编码。

In [11]:
# map按字典翻译来源名；未登记来源保留原编码，避免变成无法追溯的空值。
jobs['company_name'] = jobs['source'].map(COMPANIES).fillna(jobs['source'])
# 只增加_clean列，不覆盖原文；同一清洗规则用于标题、职责和要求。
for col in ['title', 'description', 'requirement']:
    # map将函数应用到每个单元格；clean_text同时处理空值、HTML和多余空白。
    jobs[col + '_clean'] = jobs[col].map(clean_text)
# &按元素做逻辑“且”；只有职责和要求同时为空才标记全部正文缺失。
jobs['text_missing'] = jobs['description_clean'].eq('') & jobs['requirement_clean'].eq('')
jobs['title_missing'] = jobs['title_clean'].eq('')
# zip逐行配对城市列表与备用城市名，兼容列表含字典、字符串内含多个城市等情况。
jobs['locations'] = [normalize_locations(a, b) for a, b in zip(jobs['city_list'], jobs['city_name'])]
# 先标地点层级再筛城市；“广东省”不能与“深圳”放在同一个城市排行榜。
jobs['cities'] = jobs['locations'].map(lambda xs: [x for x in xs if location_level(x) == '城市候选'])
# len计算每条职位支持的不同城市数；它不是该职位招聘的人数。
jobs['city_count'] = jobs['cities'].map(len)
# mixed尝试多种日期格式，coerce将失败值转成NaT；utc统一时区，不代表发布时间原本来自UTC。
jobs['publish_datetime'] = pd.to_datetime(jobs['publish_time'], errors='coerce', utc=True, format='mixed')

# explode把列表展开为多行；此时分析单位变成“职位—地点”，不能再把行数当职位数。
location_audit = jobs[['job_key', 'source', 'locations']].explode('locations').dropna(subset=['locations'])
location_audit['level'] = location_audit['locations'].map(location_level)
# nunique统计不同职位而不是展开行数；地点标签与层级一起保留以便审核。
location_counts = location_audit.groupby(['locations', 'level'], as_index=False).agg(关联数=('job_key', 'nunique'))
display(location_counts.sort_values('关联数', ascending=False).head(20))
print('人工核验候选：')
display(location_counts.loc[location_counts['level'].ne('城市候选')].sort_values('关联数', ascending=False).head(20))

,locations,level,关联数
60,北京,城市候选,19363
16,上海,城市候选,13076
196,杭州,城市候选,5892
258,深圳,城市候选,4952
150,广州,城市候选,1510
170,成都,城市候选,886
73,合肥,城市候选,621
322,西安,城市候选,311
205,武汉,城市候选,298
144,广东省,省级或地区,213


人工核验候选：


,locations,level,关联数
144,广东省,省级或地区,213
126,山东省,省级或地区,49
210,江苏省,省级或地区,48
382,香港特别行政区,区县待核验,45
335,辽宁省,省级或地区,40
263,湖北省,省级或地区,39
91,四川省,省级或地区,39
218,江西省,省级或地区,35
246,浙江省,省级或地区,32
229,河南省,省级或地区,30


## 4A. 【拓展】一对多关联为什么会把职位数“变多”？

一个职位可以关联多个组织。把组织关系直接并入明细后，表格可能变长，
却没有产生新职位。`merge(validate=...)`用来检查关联基数：
`one_to_one`是一对一，`one_to_many`是一对多，`many_to_many`可能造成笛卡尔膨胀。

如果只需要“组织关系条数”，先聚合关系表再关联，才能保持一行一职位。

In [12]:
# mode=ro强制只读；with管理连接事务作用域，查询不会修改原始采集数据。
with sqlite3.connect(snapshot.as_uri() + '?mode=ro', uri=True) as conn:
    organizations = pd.read_sql_query('SELECT * FROM job_organizations', conn)

# 关联键采用同一空值约定；只修正演示副本，不改数据库。
for column in KEY:
    organizations[column] = organizations[column].fillna('').astype(str)
left_jobs = jobs[KEY + ['job_key']].copy()
expanded_join = left_jobs.merge(organizations, on=KEY, how='left', validate='one_to_many')

# 先把关系表聚合为唯一职位键，再关联，得到可分析的单行特征。
organization_counts = organizations.groupby(KEY, dropna=False).size().rename('organization_count').reset_index()
safe_join = left_jobs.merge(organization_counts, on=KEY, how='left', validate='one_to_one')
safe_join['organization_count'] = safe_join['organization_count'].fillna(0).astype(int)
display(pd.DataFrame({'指标': ['原职位行数', '直接关联后的行数', '直接关联后的不同职位数', '聚合后关联行数'],
                      '数值': [len(left_jobs), len(expanded_join), expanded_join.job_key.nunique(), len(safe_join)]}))
assert expanded_join.job_key.nunique() == len(jobs)
assert len(safe_join) == len(jobs)
csv_export(safe_join[['job_key', 'organization_count']], '01_organization_join_demo')

,指标,数值
0,原职位行数,40661
1,直接关联后的行数,40950
2,直接关联后的不同职位数,40661
3,聚合后关联行数,40661


## 4B. 【主线练习】故意放入脏数据，验证你是否理解清洗

以下是人工构造的测试数据，不混入真实职位。请先指出：重复、无法解析、类型错误、
缺失分别位于哪里，再运行参考检查。不要只凭真实数据本次“恰好没错”就认为代码可靠。

In [13]:
# 两行共享同一演示ID；还有坏JSON、合法但不是对象的JSON、空payload。
dirty_demo = pd.DataFrame({'id': ['A', 'A', 'B', 'C', 'D'],
                           'payload': ['{"title":"分析师"}', '{"title":null}', '{broken', '[]', None]})
def inspect_demo_json(value):
    try:
        obj = json.loads(value)
        if not isinstance(obj, dict):
            return 'JSON合法但不是职位对象'
        return '标题缺失' if not obj.get('title') else '可用'
    except (ValueError, TypeError):
        return '无法解析或缺失'

dirty_demo['问题'] = dirty_demo['payload'].map(inspect_demo_json)
dirty_demo['重复ID'] = dirty_demo.duplicated('id', keep=False)
display(dirty_demo)
assert dirty_demo['重复ID'].sum() == 2
assert dirty_demo['问题'].eq('无法解析或缺失').sum() == 2
assert dirty_demo['问题'].eq('JSON合法但不是职位对象').sum() == 1

,id,payload,问题,重复ID
0,A,"{""title"":""分析师""}",可用,True
1,A,"{""title"":null}",标题缺失,True
2,B,{broken,无法解析或缺失,False
3,C,[],JSON合法但不是职位对象,False
4,D,None,无法解析或缺失,False


## 5. 数据质量画像与导出

分来源检查字段覆盖率，避免把“解析器没有提供学历”误解成“公司不看学历”。
主表保持一行一条职位；多城市分析下一课再展开。
Parquet 保存列表类型和空值，比把嵌套对象直接塞进 CSV 更适合课间传递。

In [14]:
# 命名聚合的格式为“新列名=(原列名,聚合函数)”；布尔列取mean就是True所占比例。
quality = jobs.groupby('company_name').agg(
    岗位数=('job_key', 'size'), 标题缺失率=('title_missing', 'mean'),
    文本全缺失率=('text_missing', 'mean'), 有城市候选率=('city_count', lambda s: s.gt(0).mean()),
    学历字段有值率=('required_degree', lambda s: s.fillna('').astype(str).str.strip().ne('').mean()),
    发布日期可解析率=('publish_datetime', lambda s: s.notna().mean()),
)
display(quality.round(3))
# 把清洗前后数量和处理动作写入机器可读日志，报告中的口径才能回到代码核验。
cleaning_log = {
    'raw_rows': len(raw), 'clean_rows': len(jobs), 'rejected_json_rows': len(rejected),
    'duplicate_identity_rows': int(jobs.duplicated(KEY).sum()),
    'identity_mismatches': identity_mismatches,
    'title_missing_rows': int(jobs['title_missing'].sum()),
    'text_missing_rows': int(jobs['text_missing'].sum()),
    'no_city_candidate_rows': int(jobs['city_count'].eq(0).sum()),
    'missing_publish_time_rows': int(jobs['publish_time'].fillna('').astype(str).str.strip().eq('').sum()),
    'unparseable_nonempty_publish_time_rows': int((
        jobs['publish_time'].fillna('').astype(str).str.strip().ne('') & jobs['publish_datetime'].isna()).sum()),
    'unparsed_or_missing_publish_time_rows': int(jobs['publish_datetime'].isna().sum()),
    'location_associations_by_level': location_audit['level'].value_counts().to_dict(),
    'actions': ['隔离坏JSON', '验证复合身份键', '原文保留并新增清洗列', '城市别名映射与地点层级标记',
                '保留多城市', '薪资学历不填0', '复杂字段JSON序列化以稳定跨课传递'],
}
write_json(REPORTS / 'cleaning_log.json', cleaning_log)
write_json(REPORTS / 'rejected_json.json', rejected)
csv_export(quality.reset_index(), '01_source_quality')
csv_export(location_counts, '01_location_audit')

# 复杂列统一存 JSON 字符串，避免不同来源结构冲突；下课只解码所需列表。
# 导出专用副本用于序列化复杂对象；内存里的jobs仍保留可操作的列表。
export = jobs.copy()

,岗位数,标题缺失率,文本全缺失率,有城市候选率,学历字段有值率,发布日期可解析率
company_name,,,,,,
OPPO,273,0.0,0.000,0.996,0.546,1.000
京东,2025,0.0,0.000,0.623,0.000,1.000
哔哩哔哩,872,0.0,0.001,1.000,0.000,1.000
字节跳动,17053,0.0,0.000,1.000,0.000,1.000
小红书,1299,0.0,0.000,0.999,0.000,1.000
快手,2563,0.0,0.000,1.000,0.000,0.101
拼多多,767,0.0,0.000,0.986,0.000,1.000
携程,534,0.0,0.000,0.957,0.000,1.000
滴滴,1808,0.0,0.000,0.928,0.320,1.000


In [15]:
# 遍历导出副本，检查列内是否有字典/列表；复杂列统一JSON编码，标量列仍保留原类型。
for col in export.columns:
    # any表示这一列至少有一个复杂对象；不要求每行都非空或具有完全相同的结构。
    if export[col].map(lambda x: isinstance(x, (list, dict))).any():
        export[col] = export[col].map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else None)
# Parquet保留数值/日期类型并压缩存储；index=False不额外保存无业务含义的行索引。
export.to_parquet(DATA / 'processed' / 'jobs_clean.parquet', index=False)
jobs[['job_key', 'source', 'title_clean', 'city_name', 'recruitment_type', 'job_url']].to_csv(
    DATA / 'processed' / 'jobs_preview.csv', index=False, encoding='utf-8-sig')
# 数量守恒：清洗结果加隔离记录必须回到原始总数，不允许静默丢行。
assert len(jobs) + len(rejected) == len(raw)
assert jobs['job_key'].is_unique
display(pd.Series(cleaning_log))

raw_rows                                                                                             40661
clean_rows                                                                                           40661
rejected_json_rows                                                                                       0
duplicate_identity_rows                                                                                  0
identity_mismatches                            {'source': 0, 'job_id': 0, 'batch_id': 0, 'source_site': 0}
title_missing_rows                                                                                       0
text_missing_rows                                                                                        3
no_city_candidate_rows                                                                                 972
missing_publish_time_rows                                                                             6470
unparseable_nonempty_publish_time_row

## 课堂练习与验收

1. 查看 `raw.iloc[0]['payload']` 的类型，再对比 `json.loads(...)` 的类型。
2. 为什么只用 `job_id` 去重不可靠？请给出两家公司可能使用同一 ID 的例子。
3. 找出城市数最多的 5 条职位，检查原始城市字段和清洗后列表。
4. 从质量表选择一个缺失率高的字段，解释为什么它不适合直接跨公司比较。

**参考解答思路**：原始 payload 是字符串、解析后是字典；职位身份包含来源、渠道、批次与站点；
多城市不能压成一个随意选择的城市；缺失可能来自接口或解析覆盖差异。

**验收**：原始行数=清洗行数+隔离行数，复合键唯一，原文仍可回溯，导出成功。
下一课读取 `data/processed/jobs_clean.parquet`。

In [16]:
display(jobs.sort_values('city_count', ascending=False)[['source', 'title_clean', 'city_name', 'cities']].head(5))
print('第1课完成：', DATA / 'processed' / 'jobs_clean.parquet')

,source,title_clean,city_name,cities
28908,meituan,进场零售-商场CKA,广州市、武汉市、安庆市、亳州市、长沙市、常州市、池州市、滁州市、鄂州市、恩施土家族苗族自治州、抚州市、阜阳市、赣州市、贵阳...,"[广州, 武汉, 安庆, 亳州, 长沙, 常州, 池州, 滁州, 鄂州, 抚州, 阜阳, 赣州, 贵阳, 杭州, 合肥, ..."
29277,meituan,营销顾问/商务拓展,厦门市、福州市、杭州市、无锡市、宁波市、北京市、温州市、苏州市、天津市、惠州市、石家庄市、上海市、成都市、长春市、珠海市、...,"[厦门, 福州, 杭州, 无锡, 宁波, 北京, 温州, 苏州, 天津, 惠州, 石家庄, 上海, 成都, 长春, 珠海,..."
28526,meituan,医药健康-城市直营中心-SMB,北京市、呼和浩特市、沈阳市、大连市、哈尔滨市、南京市、扬州市、杭州市、宁波市、温州市、嘉兴市、绍兴市、金华市、芜湖市、福州...,"[北京, 呼和浩特, 沈阳, 大连, 哈尔滨, 南京, 扬州, 杭州, 宁波, 温州, 嘉兴, 绍兴, 金华, 芜湖, 福..."
28959,meituan,火石-酒店项目-大客户销售,北京市、天津市、沈阳市、大连市、上海市、南京市、南通市、常州市、无锡市、苏州市、杭州市、宁波市、嘉兴市、合肥市、福州市、厦...,"[北京, 天津, 沈阳, 大连, 上海, 南京, 南通, 常州, 无锡, 苏州, 杭州, 宁波, 嘉兴, 合肥, 福州, ..."
29372,meituan,门店管理储备岗,杭州市、宁波市、无锡市、北京市、苏州市、天津市、上海市、石家庄市、成都市、珠海市、济南市、青岛市、广州市、深圳市、武汉市、...,"[杭州, 宁波, 无锡, 北京, 苏州, 天津, 上海, 石家庄, 成都, 珠海, 济南, 青岛, 广州, 深圳, 武汉,..."


第1课完成： /Users/luosen/WorkBuddy/JobsCrawler/data_analysis202612/data/processed/jobs_clean.parquet
